# AGFB Generator Visual Check

This notebook is a small visual smoke test for the analytic gradient field generators. It is meant to answer three questions quickly. First, can each generator be called with a short, readable example. Second, does the returned intensity image look like the intended synthetic field. Third, do the returned gradient channels have the expected shape and sign pattern.

The simplest pattern is `frame = generator(...)`, then `show_image(frame.I[0])`. Each generator returns `frame.I` with shape `(B, H, W)` and `frame.g` with shape `(B, 2, H, W)`. The first gradient channel is `g_x`, and the second gradient channel is `g_y`.

## Setup

Run this cell first. It imports the generators, imports the notebook display helpers, fixes a shared image size, and initializes the notebook display settings. Keeping this setup in one cell makes the later examples short enough that the generator call itself stays visible.

In [ ]:
# ruff: noqa: E402

# math is used only to write angles in degrees before converting to radians.
import importlib
import math

# torch is used for batched inputs, coefficient tensors, and tensor checks.
import torch

# Reload the local package so a long-running notebook kernel sees new exports.
import agfb_generators as _agfb_generators

importlib.reload(_agfb_generators)

# Each generator returns a Frame with an intensity image and analytic gradients.
from agfb_generators import (
    anisotropic_blob,
    asymmetric_ridge,
    chirp,
    curved_arc,
    curved_ridge,
    finite_ramp,
    gabor_packet,
    gaussian_blob,
    gaussian_ridge,
    junction_mask,
    mach_band,
    polynomial,
    roof_profile,
    sinusoid,
    smoothed_bar,
    smoothed_l_junction,
    smoothed_ramp,
    smoothed_step,
    smoothed_t_junction,
    smoothed_x_junction,
    smoothed_y_junction,
    vessel_bifurcation,
    vessel_bifurcation_truth,
    vessel_crossing,
    vessel_crossing_truth,
)

# Notebook helpers keep display setup and color mapping out of the examples.
from agfb_generators.notebook import set_color_scheme, setup_notebook, show_color_scheme, show_image

# All examples use the same canvas so the results are easy to compare.
H, W = 160, 160  # Height and width of the generated images.
setup_notebook(height=H, width=W)

## Color Scheme

This block defines the colors used by `show_image`. The keys describe the kind of image being displayed. `intensity` is for scalar field values, `magnitude` is for nonnegative gradient magnitudes, `signed` is for values that can be negative or positive, and `mask` is for binary or categorical overlays. The commented blocks show complete replacement schemes, so changing the notebook colors only requires swapping one dictionary.

In [ ]:
# Each ramp is a list of normalized positions and colors.
COLOR_SCHEME = {
    "intensity": [(0.0, "#000000"), (0.55, "#73000A"), (1.0, "#FFFFFF")],
    "magnitude": [(0.0, "#000000"), (0.72, "#A49137"), (1.0, "#FFFFFF")],
    "signed": [(0.0, "#CC2E40"), (0.5, "#FFFFFF"), (1.0, "#466A9F")],
    "mask": [(0.0, "#000000"), (1.0, "#CED318")],
}

# Alternative grayscale scheme.
# COLOR_SCHEME = {
#     "intensity": [(0.0, "#000000"), (1.0, "#FFFFFF")],
#     "magnitude": [(0.0, "#000000"), (1.0, "#FFFFFF")],
#     "signed": [(0.0, "#000000"), (0.5, "#A2A2A2"), (1.0, "#FFFFFF")],
#     "mask": [(0.0, "#000000"), (1.0, "#FFFFFF")],
# }

# Alternative high-contrast warm/cool scheme.
# COLOR_SCHEME = {
#     "intensity": [(0.0, "#000000"), (0.5, "#73000A"), (1.0, "#FFF2E3")],
#     "magnitude": [(0.0, "#000000"), (0.65, "#65780B"), (1.0, "#FFFFFF")],
#     "signed": [(0.0, "#1F414D"), (0.5, "#FFFFFF"), (1.0, "#CC2E40")],
#     "mask": [(0.0, "#000000"), (1.0, "#CED318")],
# }

# Apply the scheme once, then show a small legend so the mapping is visible.
set_color_scheme(COLOR_SCHEME)
show_color_scheme(COLOR_SCHEME)

## Getting `g_x` And `g_y`

The generators return the analytic gradient next to the intensity image. `frame.g` stores both channels together with shape `(B, 2, H, W)`. The channel order is fixed. `frame.g[:, 0]` is the horizontal derivative `g_x`, and `frame.g[:, 1]` is the vertical derivative `g_y`. The `frame.gx` and `frame.gy` aliases expose the same tensors with names that are easier to read.

In [ ]:
# Build one field so we can inspect the returned tensors.
frame = gaussian_blob(H, W, scale_sigma=8, center_x=-10, center_y=8)

# Direct channel access. The leading 0 selects the first batch item.
gx = frame.g[0, 0]
gy = frame.g[0, 1]

# Named aliases. These should match the direct channel access above.
gx_alias = frame.gx[0]
gy_alias = frame.gy[0]

# Gradient magnitude is not returned by the generator. It is computed here for display.
gmag = torch.sqrt(gx**2 + gy**2)

# The final expression prints shapes and alias checks in the notebook output.
(
    frame.I.shape,
    frame.g.shape,
    gx.shape,
    gy.shape,
    torch.allclose(gx, gx_alias),
    torch.allclose(gy, gy_alias),
)

In [ ]:
# Intensity uses the scalar-field color ramp.
show_image(frame.I[0], "Gaussian Blob Intensity")

# Signed derivatives use the diverging ramp so negative and positive values differ.
show_image(gx, "Gaussian Blob Horizontal Gradient", signed=True)
show_image(gy, "Gaussian Blob Vertical Gradient", signed=True)

# Magnitude is nonnegative, so it uses the magnitude ramp.
show_image(gmag, "Gaussian Blob Gradient Magnitude", kind="magnitude")

## Current Generators

Each cell below follows the same pattern. Pick parameters, call one generator, and display the first batch item with a descriptive title. When a new generator is added to the library, the notebook only needs one more cell in this section. The helper functions and color scheme do not need to change.

In [ ]:
# Smoothed step creates a single edge with a finite transition width.
frame = smoothed_step(H, W, angle_rad=math.radians(30), edge_sigma=4)

# frame.I[0] selects the first item in the batch.
show_image(frame.I[0], "Example Of Smoothed Step Generator")

In [ ]:
# curved_arc renders a smoothed circular boundary; shifting the center crops it into an arc.
frame = curved_arc(H, W, radius=200, center_x=-208, center_y=18, edge_sigma=0.5)

show_image(frame.I[0], "Example Of Curved Arc Generator")

In [ ]:
# Sinusoid creates a smooth periodic field with known orientation and frequency.
frame = sinusoid(H, W, spatial_frequency=0.05, angle_rad=math.radians(30))

show_image(frame.I[0], "Example Of Sinusoid Generator")

In [ ]:
# Gaussian blob creates a localized smooth peak.
frame = gaussian_blob(H, W, scale_sigma=8, center_x=-10, center_y=8)

show_image(frame.I[0], "Example Of Gaussian Blob Generator")

In [ ]:
# Gaussian ridge creates a long smooth ridge with a controllable direction.
frame = gaussian_ridge(H, W, width_sigma=4, angle_rad=math.radians(20))

show_image(frame.I[0], "Example Of Gaussian Ridge Generator")

In [ ]:
# Smoothed bar creates two nearby smoothed edges around a finite-width region.
frame = smoothed_bar(H, W, bar_width=32, angle_rad=math.radians(15), edge_sigma=4)

show_image(frame.I[0], "Example Of Smoothed Bar Generator")

In [ ]:
# coefficients has shape (B, x_terms, y_terms). This example uses one batch item.
coefficients = torch.zeros(1, 4, 4)

# Set a few low-order terms so the field has visible tilt and curvature.
coefficients[0, 1, 0] = 0.3
coefficients[0, 0, 1] = -0.2
coefficients[0, 2, 1] = 0.05
coefficients[0, 1, 2] = -0.04

# coordinate_scale controls the coordinate normalization used by the polynomial.
frame = polynomial(H, W, coefficients=coefficients, coordinate_scale=64)

show_image(frame.I[0], "Example Of Polynomial Generator")

## Transition Generators

These fields test finite transitions rather than isolated edges. The ramp variants control transition width, the roof profile creates a triangular ridge-like edge response, and the Mach-band field adds paired shoulders around a smoothed ramp.

In [ ]:
# Finite ramp creates a linear transition with flat plateaus on both sides.
frame = finite_ramp(H, W, ramp_width=64, angle_rad=math.radians(0))

show_image(frame.I[0], "Example Of Finite Ramp Generator")

In [ ]:
# Smoothed ramp is the Gaussian-smoothed version of the finite ramp.
frame = smoothed_ramp(H, W, ramp_width=64, angle_rad=math.radians(20), edge_sigma=4)

show_image(frame.I[0], "Example Of Smoothed Ramp Generator")

In [ ]:
# Roof profile creates a triangular profile with a sharp peak at the center.
frame = roof_profile(H, W, roof_width=70, angle_rad=math.radians(10))

show_image(frame.I[0], "Example Of Roof Profile Generator")

In [ ]:
# Mach band adds bright and dark shoulders around a smoothed ramp.
frame = mach_band(
    H,
    W,
    ramp_width=70,
    angle_rad=math.radians(0),
    edge_sigma=4,
    shoulder_amplitude=0.12,
    shoulder_sigma=4,
)

show_image(frame.I[0], "Example Of Mach Band Generator")

## Scale And Frequency Generators

These fields stress filters that estimate scale, orientation, and local frequency. The anisotropic blob has two independent scales, the chirp changes frequency across the image, and the Gabor packet combines a sinusoid with a localized envelope.

In [ ]:
# Anisotropic blob creates an oriented Gaussian peak with two scale parameters.
frame = anisotropic_blob(
    H,
    W,
    length_sigma=20,
    width_sigma=7,
    angle_rad=math.radians(30),
    center_x=-8,
    center_y=10,
)

show_image(frame.I[0], "Example Of Anisotropic Blob Generator")

In [ ]:
# Chirp creates a sinusoid whose frequency changes along one direction.
frame = chirp(H, W, base_frequency=0.015, frequency_slope=0.00035, angle_rad=math.radians(0))

show_image(frame.I[0], "Example Of Chirp Generator", signed=True)

In [ ]:
# Gabor packet localizes an oriented sinusoid with an anisotropic envelope.
frame = gabor_packet(
    H,
    W,
    carrier_frequency=0.055,
    angle_rad=math.radians(25),
    envelope_length_sigma=36,
    envelope_width_sigma=14,
)

show_image(frame.I[0], "Example Of Gabor Packet Generator", signed=True)

## Junction Generators

Junction generators compose half-infinite smoothed bars with a smooth union. They expose L-, T-, Y-, and X-shaped structures while keeping the generator return type as a plain analytic `Frame`.

In [ ]:
# Smoothed L junction creates two perpendicular half-bar arms.
frame = smoothed_l_junction(H, W, arm_width=18, angle_rad=math.radians(0), edge_sigma=3)

show_image(frame.I[0], "Example Of Smoothed L Junction Generator")

In [ ]:
# Smoothed T junction creates a trunk with two opposite cross arms.
frame = smoothed_t_junction(H, W, arm_width=18, angle_rad=math.radians(0), edge_sigma=3)

show_image(frame.I[0], "Example Of Smoothed T Junction Generator")

In [ ]:
# Smoothed Y junction creates three evenly spaced arms.
frame = smoothed_y_junction(H, W, arm_width_px=16, theta_rad=math.radians(-90), sigma_e=3)

show_image(frame.I[0], "Example Of Smoothed Y Junction Generator")

In [ ]:
# Smoothed X junction creates four crossing half-bar arms.
frame = smoothed_x_junction(H, W, arm_width=15, angle_rad=math.radians(45), edge_sigma=3)

show_image(frame.I[0], "Example Of Smoothed X Junction Generator")

In [ ]:
# Junction truth helper marks the local scoring neighborhood around the center.
mask = junction_mask(H, W, radius_px=16)

show_image(mask.float(), "Example Of Junction Truth Mask", kind="mask")

## Ridge And Vessel Generators

These fields extend the simple Gaussian ridge into asymmetric profiles, curved centerlines, crossings, and bifurcations. Vessel truth helpers provide semantic masks and labels for later metric code without changing the `Frame` return type.

In [ ]:
# Asymmetric ridge uses different widths on each side of the centerline.
frame = asymmetric_ridge(H, W, negative_sigma=5, positive_sigma=14, angle_rad=math.radians(20))

show_image(frame.I[0], "Example Of Asymmetric Ridge Generator")

In [ ]:
# Curved ridge bends a Gaussian ridge with a quadratic centerline model.
frame = curved_ridge(H, W, width_sigma=5, angle_rad=math.radians(0), curvature=0.006)

show_image(frame.I[0], "Example Of Curved Ridge Generator")

In [ ]:
# Vessel crossing sums two Gaussian vessel branches at different orientations.
frame = vessel_crossing(
    H,
    W,
    branch_a_width_sigma=5,
    branch_b_width_sigma=4,
    branch_a_normal_angle_rad=math.radians(25),
    branch_b_normal_angle_rad=math.radians(115),
    branch_a_amplitude=0.85,
    branch_b_amplitude=0.9,
)

show_image(frame.I[0], "Example Of Vessel Crossing Generator")

In [ ]:
# Vessel bifurcation gates three Gaussian branches around one junction point.
frame = vessel_bifurcation(
    H,
    W,
    trunk_width_sigma=5,
    left_width_sigma=4,
    right_width_sigma=4,
    trunk_tangent_angle_rad=math.radians(-90),
    left_tangent_angle_rad=math.radians(35),
    right_tangent_angle_rad=math.radians(145),
    branch_gate_sigma=4,
)

show_image(frame.I[0], "Example Of Vessel Bifurcation Generator")

In [ ]:
# Vessel truth helpers expose masks and branch labels for metrics.
crossing_truth = vessel_crossing_truth(
    H,
    W,
    branch_a_width_sigma=5,
    branch_b_width_sigma=4,
    branch_a_normal_angle_rad=math.radians(25),
    branch_b_normal_angle_rad=math.radians(115),
)
bifurcation_truth = vessel_bifurcation_truth(
    H,
    W,
    trunk_width_sigma=5,
    left_width_sigma=4,
    right_width_sigma=4,
    trunk_tangent_angle_rad=math.radians(-90),
    left_tangent_angle_rad=math.radians(35),
    right_tangent_angle_rad=math.radians(145),
    branch_gate_sigma=4,
)

show_image(
    crossing_truth["centerline_mask"].float(), "Vessel Crossing Centerline Truth", kind="mask"
)
show_image(
    (bifurcation_truth["branch_label"] > 0).float(), "Vessel Bifurcation Branch Truth", kind="mask"
)

## A Batch Example

Most generators accept scalar parameters or batched tensor parameters. Tensor parameters make it possible to generate several related fields with one call. The batch dimension appears first in `frame.I` and `frame.g`, so `frame.I[0]`, `frame.I[1]`, and `frame.I[2]` display the three generated examples.

In [ ]:
# Tensor parameters create three smoothed steps in one generator call.
frame = smoothed_step(
    H,
    W,
    angle_rad=torch.tensor([0.0, math.radians(30), math.radians(60)]),
    edge_sigma=torch.tensor([2.0, 4.0, 6.0]),
)

# Each batch item has the same shape but different edge parameters.
show_image(frame.I[0], "Batched Smoothed Step Item 0")
show_image(frame.I[1], "Batched Smoothed Step Item 1")
show_image(frame.I[2], "Batched Smoothed Step Item 2")